In [ ]:
# mount to google drive
import os
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Colab Notebooks/StackedDataAugmentation


Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/StackedDataAugmentation


# Create Data Augmentation Permutations

In [ ]:
from itertools import permutations
# creating list of all possible permutation

data_augmentations = ['GN', 'TT', 'GAN']
# GN = Random Gaussian Noise
# TT = Time Transformation
# GAN = GAN models

all_permutations = []
for r in range(0, len(data_augmentations)+1):
  perms = list(permutations(data_augmentations, r))
  all_permutations.extend(perms)

In [ ]:
for p in all_permutations:
  print(p)

()
('GN',)
('TT',)
('GAN',)
('GN', 'TT')
('GN', 'GAN')
('TT', 'GN')
('TT', 'GAN')
('GAN', 'GN')
('GAN', 'TT')
('GN', 'TT', 'GAN')
('GN', 'GAN', 'TT')
('TT', 'GN', 'GAN')
('TT', 'GAN', 'GN')
('GAN', 'GN', 'TT')
('GAN', 'TT', 'GN')


# Importing Real Training and Validation Data

In [ ]:
import pandas as pd
# import training data

# path to the training file pkl
train_df_path = '/content/drive/MyDrive/Colab Notebooks/StackedDataAugmentation/physionetdata/eegmmidb_train_df.pkl'
val_df_path = '/content/drive/MyDrive/Colab Notebooks/StackedDataAugmentation/physionetdata/eegmmidb_val_df.pkl'

In [ ]:
from GANTraining.EEGMMIDBDatasetLoaderV2 import EEGMMIDBDataset
from torch.utils.data import DataLoader

train_dataset = EEGMMIDBDataset(pickle_path=train_df_path, purpose='eegnet', onehot=True)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

val_dataset = EEGMMIDBDataset(pickle_path=val_df_path, purpose='eegnet', onehot=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True)

# Importing GAN model Generator

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch
from GANTraining.GANModels import *
import numpy as np
import os

In [ ]:
# Define model paths
model_paths = {
    'left_hand': '/content/drive/MyDrive/Colab Notebooks/StackedDataAugmentation/GANTraining/logs/GAN_left_hand_exp_2025_07_06_20_28_14/Model/checkpoint_epoch111',
    'right_hand': '/content/drive/MyDrive/Colab Notebooks/StackedDataAugmentation/GANTraining/logs/GAN_right_hand_exp_2025_07_06_20_53_07/Model/checkpoint_epoch113',
    'both_hands': '/content/drive/MyDrive/Colab Notebooks/StackedDataAugmentation/GANTraining/logs/GAN_both_hands_exp_2025_07_06_21_18_14/Model/checkpoint_epoch111',
    'both_feet': '/content/drive/MyDrive/Colab Notebooks/StackedDataAugmentation/GANTraining/logs/GAN_both_feet_exp_2025_07_06_21_43_17/Model/checkpoint_epoch111',
    'rest_model': '/content/drive/MyDrive/Colab Notebooks/StackedDataAugmentation/GANTraining/logs/GAN_rest_exp_2025_07_07_01_53_42/Model/checkpoint_epoch28'
}

In [ ]:
%cd GANTraining

/content/drive/MyDrive/Colab Notebooks/StackedDataAugmentation/GANTraining


In [ ]:
from GANTraining.SyntheticGANDataset import SyntheticGANEEGDataset
# # Instantiate dataset and dataloader
# practice_synthetic_dataset = SyntheticGANEEGDataset(model_paths, sample_size=300, purpose='eegnet', onehot=True)
# practice_synthetic_loader = DataLoader(practice_synthetic_dataset, batch_size=32, shuffle=True)

# Checking Shape of Data

In [ ]:
%cd ..


/content/drive/MyDrive/Colab Notebooks/StackedDataAugmentation


In [ ]:
# import data augmentation techniques
from EEGNoiseAndTimeTransformations import apply_time_shift, apply_gaussian_noise, plot_single_eeg_sample, dataloader_to_numpy

In [ ]:
# # checking shape of real train_loader
# for batch in train_loader:
#   print('shape of train_loader')
#   X_train, y_train = batch
#   print(f"Train batch shape: {X_train.shape}")
#   print(f"Labels shape: {y_train.shape},")
#   break

# # checking shape of synthetic_loader data
# for batch in practice_synthetic_loader:
#     X_syn, y_syn = batch
#     print(f"Synthetic batch shape: {X_syn.shape}")
#     print(f"Labels shape: {y_syn.shape},")
#     break

# Training 64 EEGNets

In [ ]:
# getting validation dataset
X_val, y_val = dataloader_to_numpy(val_loader)

In [ ]:
import pandas as pd
results = []

In [ ]:
save_dir = 'saved_eegnet_models'
# os.makedirs(save_dir, exist_ok=True)


In [ ]:
if os.path.exists(f'{save_dir}/EEGNET_50pct_GAN.h5'):
  print("Directory already exists")
else:
  os.makedirs(save_dir)
  print("Directory created")

FileExistsError: [Errno 17] File exists: 'saved_eegnet_models'

In [ ]:
# from tqdm import tqdm
# from EEGModels import EEGNet
# # training EEGNet loop
# list_of_percents = [0.25, 0.50, 0.75, 1.0]

# for permutation in tqdm(all_permutations, desc='Permutation %'):
#   print(f"Permutation: {permutation}")
#   for percent in tqdm(list_of_percents, desc=f"Testing aug %"):
#     print(f"Percent of data augmenting: {str(percent)}")

#     # get original (from dataloader) data
#     curr_X_train, curr_y_train = dataloader_to_numpy(train_loader)

#     # calculate amount of data to augment
#     amount_to_augment = int(percent * len(curr_X_train))
#     print(f"Amount of data to augment: {amount_to_augment}")

#     # see if model has been trained already
#     model_name = f'EEGNET_{int(percent*100)}pct_{"_".join(permutation)}.h5'
#     model_path = os.path.join(save_dir, model_name)
#     if os.path.exists(model_path):
#       print(f"Model {model_name} already exists. Skipping augmentation.")
#       continue

#     # perform augmentations
#     for aug in permutation:


#       if aug == 'GAN':
#         # divide amount_to_augment by 5 so that all classes gan generated signals = amount_to_augment
#         GAN_amount_to_augment = int(amount_to_augment/5)

#         # get GAN data
#         synthetic_dataset = SyntheticGANEEGDataset(model_paths,
#                                                    sample_size=GAN_amount_to_augment,
#                                                    purpose='eegnet',
#                                                    onehot=True
#                                                    )
#         synthetic_loader = DataLoader(synthetic_dataset, batch_size=32, shuffle=True)

#         # convert generated signals into numpy objects to add back to dataset
#         GAN_X, GAN_y = dataloader_to_numpy(synthetic_loader)

#         # adding GAN_X, and GAN_y back to curr_X_train, and curr_y_train
#         curr_X_train = np.concatenate([curr_X_train, GAN_X], axis=0)
#         curr_y_train = np.concatenate([curr_y_train, GAN_y], axis=0)


#       else:
#         # pick amount_to_augment rows of data
#         indices = np.random.choice(len(curr_X_train), size=amount_to_augment, replace=False)

#         # get rows associated with X and y
#         X_subset = curr_X_train[indices]
#         y_subset = curr_y_train[indices]

#         # copy y_subset
#         y_augmented = y_subset.copy()

#         # apply augmentation methods
#         if aug == 'GN':
#           X_augmented = np.array([apply_gaussian_noise(X) for X in X_subset])

#         elif aug == 'TT':
#             X_augmented = np.array([apply_time_shift(X) for X in X_subset])


#         # combine X_augmented back to full dataset
#         curr_X_train = np.concatenate([curr_X_train, X_augmented], axis=0)
#         curr_y_train = np.concatenate([curr_y_train, y_augmented], axis=0)


#     # create EEGNET model
#     model = EEGNet(nb_classes=5,    # num of classes
#                   Chans=64,       # num of channels
#                   Samples=640,     # seq_len
#                   )
#     model.compile(optimizer='adam',
#                   loss='categorical_crossentropy',
#                   metrics=['accuracy'])

#     # train model
#     model.fit(curr_X_train, curr_y_train, epochs=20, batch_size=32)

#     # save model
#     model_path = os.path.join(save_dir, model_name)
#     model.save(model_path)

# Evaluating Models on Test Dataset

In [ ]:
# get path for all models
model_paths = [os.path.join(save_dir, f) for f in os.listdir(save_dir) if f.endswith('.h5')]
print(f'Number of Models: {len(model_paths)}')

In [ ]:
# loading original non-augmented training dataset
final_training_df = EEGMMIDBDataset(pickle_path=train_df_path, purpose='eegnet', onehot=True)
final_training_loader = DataLoader(final_training_df, batch_size=32, shuffle=True)

# loading test dataset
test_df_path = '/content/drive/MyDrive/Colab Notebooks/StackedDataAugmentation/physionetdata/eegmmidb_test_df.pkl'
test_dataset = EEGMMIDBDataset(pickle_path=test_df_path, purpose='eegnet', onehot=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

# load training and testing data as numpy
final_x_train, final_y_train = dataloader_to_numpy(final_training_loader)
X_test, y_test = dataloader_to_numpy(test_loader)


In [ ]:
import tensorflow as tf

# saving training, and test results to csv
results = []

for path in tqdm(model_paths, desc='loading model'):
  # load model
  model = tf.keras.models.load_model(path)

  # evaluate model on training data
  train_loss, train_acc = model.evaluate(final_X_train, final_y_train)

  # evaluate model on test data
  test_loss, test_acc = model.evaluate(X_val, y_val)

  # getting metadata from filename
  filename = os.path.basename(path)
  parts = filename.replace('.h5', '').split('_')
  percent = parts[1].replace('pct', '')
  augmentations = parts[2:]

  # appending results to list
  results.append({
      'model_name': filename,
      'percent_augmented': float(percent)/100,
      'augmentations': augmentations,
      'train_loss': train_loss,
      'train_acc': train_acc,
      'test_loss': test_loss,
      'test_acc': test_acc
  })

df = pd.DataFrame(results)
df.to_csv('eegnet_eval_results.csv', index=False)

!ls

# Flatten list to string instead of list
# df['augmentations_combined'] = df['augmentations'].apply(lambda x: '+'.join(x) if isinstance(x, list) else str(x))

df.head()

# Getting Distribution of Data

In [ ]:
from GANTraining.EEGMMIDBDatasetLoaderV2 import EEGMMIDBDataset
from torch.utils.data import DataLoader

# test data path
test_df_path = '/content/drive/MyDrive/Colab Notebooks/StackedDataAugmentation/physionetdata/eegmmidb_test_df.pkl'

# get test data dataloader
test_dataset = EEGMMIDBDataset(pickle_path=test_df_path, purpose='eegnet', onehot=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

# load test data
X_test, y_test = dataloader_to_numpy(test_loader)

In [ ]:
# get distribution of test data
import numpy as np

# Convert one-hot to class indices
y_test_labels = np.argmax(y_test, axis=1)

# Get distribution
unique, counts = np.unique(y_test_labels, return_counts=True)

# Print distribution
for label, count in zip(unique, counts):
    print(f"Class {label}: {count} samples")

# Visuals

In [ ]:
df.head()

import seaborn as sns
import matplotlib.pyplot as plt



plt.figure(figsize=(14, 6))
sns.barplot(data=df, x='percent_augmented', y='test_acc', hue='augmentations_combined')
plt.title('Test Accuracy vs. Augmentation (%) by Method')
plt.ylabel('Test Accuracy')
plt.xlabel('Percent of Data Augmented')
plt.ylim(0, 1)
plt.legend(title='Augmentation')
plt.tight_layout()
plt.show()